In [1]:
import pandas as pd 
import numpy as np
df=pd.read_csv('used_cars_data.csv')
df.head()

,S.No.,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,0,Maruti Wagon R LXI CNG,Mumbai,2010,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,NaN,1.75
1,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74


In [2]:
# remove the columns 
df.drop(['S.No.','New_Price'],axis=1,inplace=True)

In [3]:
df['Car_maker']=df['Name'].str.split().str.get(0)
df['Car_model']=df['Name'].str.split().str.get(1)

In [4]:
df['Mileage']=df['Mileage'].str.replace('km/kg','').str.replace('kmpl','').astype(float)
df['Engine']=df['Engine'].str.replace('CC','').astype(float)
df['Power']=pd.to_numeric(df['Power'].str.replace('bhp','').str.replace('null',''),errors='coerce')
df.drop(['Name'],axis=1,inplace=True)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7253 entries, 0 to 7252
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           7253 non-null   object 
 1   Year               7253 non-null   int64  
 2   Kilometers_Driven  7253 non-null   int64  
 3   Fuel_Type          7253 non-null   object 
 4   Transmission       7253 non-null   object 
 5   Owner_Type         7253 non-null   object 
 6   Mileage            7251 non-null   float64
 7   Engine             7207 non-null   float64
 8   Power              7078 non-null   float64
 9   Seats              7200 non-null   float64
 10  Price              6019 non-null   float64
 11  Car_maker          7253 non-null   object 
 12  Car_model          7253 non-null   object 
dtypes: float64(5), int64(2), object(6)
memory usage: 736.8+ KB


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


In [7]:
df.dropna(how='any',inplace=True)
x=df.drop('Price',axis=1)
y=df['Price']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=2024)
x_train,x_val,y_train,y_val=train_test_split(x_train,y_train,test_size=0.2,random_state=2024)

# Base line experiment

In [8]:
df.dropna(how='any',inplace=True)

In [9]:
df.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Car_maker,Car_model
0,Mumbai,2010,72000,CNG,Manual,First,26.60,998.0,58.16,5.0,1.75,Maruti,Wagon
1,Pune,2015,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,Hyundai,Creta
2,Chennai,2011,46000,Petrol,Manual,First,18.20,1199.0,88.70,5.0,4.50,Honda,Jazz
3,Chennai,2012,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,Maruti,Ertiga
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,Audi,A4


# Base line experiment:

# Experiment 1

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import numpy as np

from sklearn.linear_model import LinearRegression
features=['Year','Kilometers_Driven','Mileage','Engine','Power','Seats']

feature_select=ColumnTransformer(
    [('Select_Features','passthrough',features)]
)

pipeline=Pipeline(steps=[
    ('Feature_Selection',feature_select),
    ('Modeling',LinearRegression())
])
pipeline.fit(x_train,y_train)
train_prediction=pipeline.predict(x_train)
valid_prediction=pipeline.predict(x_val)

print('r^2 on train data =',r2_score(y_train,train_prediction))
print('r^2 on validation data =',r2_score(y_val,valid_prediction))

r^2 on train data = 0.6789866227765811
r^2 on validation data = 0.6829010473973434


# Experiment 2

In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import numpy as np

# Assuming x_train and y_train are already defined
features = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
categorical_features = ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Car_maker', 'Car_model']

# Column transformer to handle both numeric and categorical features
feature_select = ColumnTransformer(
    [('Select_Features', 'passthrough', features),
     ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)]
)

# Creating a pipeline with feature selection and linear regression model
pipeline = Pipeline(steps=[
    ('Feature_Selection', feature_select),
    ('Modeling', LinearRegression())
])

# Perform cross-validation
cv_scores = cross_val_score(pipeline, x_train, y_train, cv=5, scoring='r2')  # 5-fold cross-validation

# Print the R² scores for each fold and the average score
print("Cross-validation R² scores:", cv_scores)
print("Mean R² score across all folds:", np.mean(cv_scores))

# Fit the model on the full training set
pipeline.fit(x_train, y_train)

# Make predictions on the training and validation sets
train_prediction = pipeline.predict(x_train)
valid_prediction = pipeline.predict(x_val)

# Print R² scores on train and validation sets
print('R² on train data =', r2_score(y_train, train_prediction))
print('R² on validation data =', r2_score(y_val, valid_prediction))


c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found 

Cross-validation R² scores: [0.69393137 0.6874618  0.61445762 0.20965608 0.69054753]
Mean R² score across all folds: 0.5792108794276905
R² on train data = 0.679201753326945
R² on validation data = 0.6783712108879437


c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [12]:
from sklearn.ensemble import RandomForestRegressor 
import numpy as np

# Assuming x_train and y_train are already defined
features = ['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
categorical_features = ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Car_maker', 'Car_model']

# Column transformer to handle both numeric and categorical features
feature_select = ColumnTransformer(
    [('Select_Features', 'passthrough', features),
     ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)]
)

# Creating a pipeline with feature selection and linear regression model
pipeline = Pipeline(steps=[
    ('Feature_Selection', feature_select),
    ('Modeling', RandomForestRegressor())
])

# Perform cross-validation
cv_scores = cross_val_score(pipeline, x_train, y_train, cv=5, scoring='r2')  # 5-fold cross-validation

# Print the R² scores for each fold and the average score
print("Cross-validation R² scores:", cv_scores)
print("Mean R² score across all folds:", np.mean(cv_scores))

# Fit the model on the full training set
pipeline.fit(x_train, y_train)

# Make predictions on the training and validation sets
train_prediction = pipeline.predict(x_train)
valid_prediction = pipeline.predict(x_val)

# Print R² scores on train and validation sets
print('R² on train data =', r2_score(y_train, train_prediction))
print('R² on validation data =', r2_score(y_val, valid_prediction))


c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found 

Cross-validation R² scores: [0.91258818 0.90313768 0.83125477 0.93858151 0.84066413]
Mean R² score across all folds: 0.8852452545232345
R² on train data = 0.9829202281041381
R² on validation data = 0.9043384156911264


c:\Users\User\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [13]:
import joblib

# Save the pipeline after fitting the model
joblib.dump(pipeline,'car_price_predictor.pkl')


['car_price_predictor.pkl']

In [14]:
df.to_csv('Used.csv',index=False)